<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **6. Wi-Fi Network Performance**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)


  Remark: Note that running the underlying simulations of this notebook might take some time (a few minutes). If you cannot run the simulations, use the illustrations that are already available.
  
</div>



### 6.1 Introduction

The motivation for this notebook is to analyze the basics of the WLAN IEEE 802.11 medium access protocol in scenarios with many transmitting devices, instead of only one. When more than one station transmits data, radio resources are shared, and collisions might occur. The goal is to evaluate the performance of the underlying contention-based protocol: What is the maximum capacity, and what are the implications for the throughput with an increasing number of stations?

In [ ]:
import platform

classpath_jemula = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar"
classpath_jemula802 = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar:jemula/lib/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")



#### 6.1.1 Clone (or update) the Jemula802 Simulator 

In [ ]:
%%bash
set -e
if [ -d "jemula802/.git" ]; then
  echo "Updating existing repo jemula802 ..."
  git -C jemula802 pull --ff-only
else
  echo "Cloning repo jemula802 ..."
  git clone https://bitbucket.org/lfield/jemula802.git
fi
set -e
if [ -d "jemula/.git" ]; then
  echo "Updating existing repo jemula ..."
  git -C jemula pull --ff-only
else
  echo "Cloning repo jemula ..."
  git clone https://bitbucket.org/lfield/jemula.git
fi

#### 6.1.2 Compile jemula802 and jemula Java Code, if needed

In [ ]:
!javac -g:none -classpath $classpath_jemula -d "jemula/bin" jemula/src/plot/*.java jemula/src/kernel/*.java jemula/src/statistics/*.java
!echo Compilation of jemula done.

!javac -g:none  -classpath $classpath_jemula802 -d "jemula802/bin" jemula802/src/emulator/*.java jemula802/src/layer0_medium/*.java jemula802/src/layer1_802Phy/*.java \
 jemula802/src/layer1_80211Phy/*.java jemula802/src/layer2_802Mac/*.java jemula802/src/layer2_80211Mac/*.java \
 jemula802/src/layer2_802Algorithms/*.java jemula802/src/layer3_network/*.java jemula802/src/layer5_application/*.java \
 jemula802/src/layer4_transport/*.java jemula802/src/station/*.java jemula802/src/visualization/*.java \
 jemula802/src/util/*.java jemula802/src/gui/*.java
!echo Compilation of jemula802 done.

In [ ]:
import os

ROOT_DIR = os.getcwd()
!echo Root directory is $ROOT_DIR

#### 6.1.3 Load Evaluation Scripts for Later Use

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

os.chdir(ROOT_DIR)

In [ ]:
def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay,ylim):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.ylim(0, ylim)  # Set y-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6], '-', color='g', linewidth=4, label='Offer [Mb/s]')
    plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6], '-', color='w', linewidth=1, label='Throughput [Mb/s]')
    plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3], '-', color='r', linewidth=1, label='Average Delay [ms]')
    #plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 5], '-', color='r', linewidth=1)
    plt.legend(loc="upper left", fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(aBinSize_ms_string, aHistogram_string):
    fig, ax_distr = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))
    theHistogram = [float(x) for x in aHistogram_string.split()]
    theHistogram = [x/sum(theHistogram) for x in theHistogram]
    theNumOfBins = len(theHistogram)
    theBinSize_ms = float(aBinSize_ms_string)
    theX = [i * theBinSize_ms + theBinSize_ms/2 for i in range(theNumOfBins)] 
    theCumSum = 1-np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)
    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)  # Set x-axis limits
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay>=x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

### 6.2 Scenario "MobComp_assignment04.xml"
Note the name of the XML Scenario file below. This file, [ETH_Course/MobComp_assignment04.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04.xml), defines what will be simulated. 
The simulation results and a copy of the XML Scenario file are stored in 
"[results_MobComp_A04](results_MobComp_A04)".

>This simulation models a single-hop Wi-Fi (802.11a/g) link on channel 6 with one central receiver and 36 transmitters arranged in a circle around it. Each transmitter starts sending UDP traffic (0.75 Mb/s) at staggered times, so the offered load gradually rises to about 27 Mb/s. The run lasts 50 seconds and tracks throughput, delay, and load. All devices use 54 Mb/s, 10 dBm transmit power, omnidirectional antennas, and a 50 m coverage range. The MAC applies 802.11e EDCA with small contention windows and fragmentation of 1200-byte packets, while RTS/CTS is off. Overall, the setup creates a many-to-one uplink where more and more senders contend for the channel, showing how throughput, delay, and collisions change as network load approaches the medium’s capacity.

<center>
<a href="https://gitlab.ethz.ch/wireless/MobileComputing/-/raw/main/.admin/notebooks/picture/jemula_scenario01.png"><img src="../../.admin/notebooks/picture/jemula_scenario01.png" 
     alt="Scenario - click to open the image" 
     width="80%" 
     style="filter: grayscale(80%);"></a>
</center>

#### 6.2.1 Simulate

In [ ]:
os.chdir(ROOT_DIR)
SCENARIO01="jemula802/scenarios/ETH_Course/MobComp_assignment04.xml"
!echo Running scenario $SCENARIO01
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO01

#### 6.2.2 Plot the Results

In [ ]:
if os.path.isdir("results_MobComp_A04"):
    os.chdir("results_MobComp_A04")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay, 180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

### 6.3 Modified dot11EDCAAIFSN

The new XML Scenario file is [ETH_Course/MobComp_assignment04_modified_dot11EDCAAIFSN.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCAAIFSN.xml).
The simulation results will be stored in 
"[results_MobComp_A04_modified_dot11EDCAAIFSN](results_MobComp_A04_modified_dot11EDCAAIFSN)".

>This simulation again uses a many-to-one Wi-Fi (802.11a/g) uplink on channel 6 with one central receiver and 36 transmitters arranged on a 6 m ring. Each sender starts a 0.75 Mb/s UDP flow at staggered times, so the load rises to about 27 Mb/s over 50 seconds, while throughput, delay, and load are logged. Radios use 54 Mb/s PHY, 10 dBm power, omnidirectional antennas, and fragmentation of 1200-byte packets; RTS/CTS is disabled. The main change from the previous setup is in the EDCA contention parameters: AIFSN is increased to 8, CWmax raised to 100 (from 10), CWmin stays at 5, and TXOP is 3008 µs. This makes stations wait longer and back off over a wider range after collisions, lowering collision probability and smoothing access under heavy contention. The trade-off is higher access delay and reduced peak throughput, highlighting how conservative EDCA settings balance fairness and stability against capacity and latency.


#### 6.3.1 Load the Modified XML Scenario File

In [ ]:
os.chdir(ROOT_DIR)
SCENARIO02="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCAAIFSN.xml"
!echo Modified scenario is $SCENARIO02

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO02, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO02, lineterm='')

#for line in diff:
#    print(line)


#### 6.3.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO02

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_dot11EDCAAIFSN"):
    os.chdir("results_MobComp_A04_modified_dot11EDCAAIFSN")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay, 180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

### 6.4 Modified dot11EDCACWmin

In IEEE 802.11, `dot11EDCACWmin` is a parameter used in the Enhanced Distributed Channel Access (EDCA) mechanism, which provides Quality of Service (QoS) by assigning different priorities to different types of traffic. Each access category—voice, video, best effort, and background—has its own set of channel access parameters, including AIFSN (Arbitration Inter-Frame Space Number), CWmin (minimum contention window), CWmax (maximum contention window), and TXOP limit (the maximum time a station can hold the channel once it gains access). The parameter `dot11EDCACWmin` defines the minimum size of the contention window for a given access category. When a station wants to transmit, it senses the medium and then randomly chooses a backoff counter from the range 0 to CWmin. If a collision occurs, the contention window is doubled up to CWmax. A smaller CWmin leads to shorter average backoff times, meaning that the access category has a higher priority when contending for the channel. For example, voice traffic is typically assigned a smaller CWmin than video, best effort, or background traffic, so that voice frames are more likely to be transmitted quickly. In essence, `dot11EDCACWmin` is the parameter that determines how aggressively a given traffic category attempts to access the wireless medium, with smaller values providing higher priority.

The new XML Scenario file is [ETH_Course/MobComp_assignment04_modified_dot11EDCACWmin.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCACWmin.xml).
The simulation results will be stored in 
"[results_MobComp_A04_modified_dot11EDCACWmin](results_MobComp_A04_modified_dot11EDCACWmin)".

>This scenario again simulates a many-to-one 802.11a/g uplink with one receiver at (0,0) and 36 mobile senders arranged on a circle of about 6 m radius, each generating a UDP best-effort flow of 0.75 Mb/s toward the receiver. The flows start one by one between 2 s and 37 s, so the aggregate offered load ramps up to around 27 Mb/s. The run lasts 50 s with statistics taken every 0.5 s and written to `results_MobComp_A04_modified_dot11EDCACWmin`. PHY and MAC parameters remain the same as in the baseline: 802.11a/g on channel 6 at 2437 MHz with 10 dBm omni antennas, BER curves enabled, RTS/CTS disabled (threshold 2500 B), fragmentation enabled for the 1200-byte packets (threshold 512 B), TXOPLimit 3008 µs, and a coverage range of 50 m, supporting data rates up to 54 Mb/s. The only change is in the EDCA contention settings for AC=2, where CWmin is raised from 5 to 8 while AIFSN remains 4 and CWmax remains 10. This larger CWmin increases the initial backoff window, which lowers collision probability once many stations are active, at the cost of slightly higher access delays and potentially a small throughput reduction under lighter loads. Since CWmax is still small, backoff growth is limited, so collisions can still occur at saturation, though less bursty than with the smaller CWmin. Overall, compared to the baseline, this variant should produce smoother contention behavior, a right-shift in delay distributions, similar total throughput until the medium saturates, and somewhat fairer sharing among the many senders.


#### 6.4.1 Load the Modified XML Scenario File


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO03="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCACWmin.xml"
!echo Modified scenario is $SCENARIO03

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO03, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO03, lineterm='')

#for line in diff:
#    print(line)

#### 6.4.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO03

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_dot11EDCACWmin"):
    os.chdir("results_MobComp_A04_modified_dot11EDCACWmin")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay ,180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

### 6.5 Modify Station 10
The new XML Scenario file is [ETH_Course/MobComp_assignment04_modified_Station10only.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_Station10only.xml).
The simulation results will be stored in 
"[results_MobComp_A04_modified_Station10only](results_MobComp_A04_modified_Station10only)".

>This scenario is the same many-to-one 802.11a/g uplink as before—one fixed receiver at (0,0) and 36 mobile senders on a ~6 m circle, each running a 0.75 Mb/s UDP best-effort flow that starts sequentially from 2 s to 37 s—over a 50 s run with 0.5 s statistics written to `results_MobComp_A04_modified_Station10only`. PHY/MAC remain standard: channel 6 at 2437 MHz, 10 dBm omni antennas, BER curves enabled, RTS/CTS effectively off (threshold 2500 B), fragmentation active for 1200-byte packets (threshold 512 B), TXOPLimit 3008 µs, 50 m coverage, rates up to 54 Mb/s. All stations use EDCA AC=2 with AIFSN=4 and CWmin=5, CWmax=100—except the node with MACAddress 10 (starting at 10 s, position ≈(1.04, 5.91)), which is made more aggressive with AIFSN=2 and CWmin=1 (CWmax still 100). Two senders (start 8 s and 10 s) have delay/offer evaluation enabled, otherwise eval is throughput-centric. Functionally, the “station-10-only” tweak reduces its contention delay and boosts its access probability once the network is busy, so you should see higher airtime and throughput for that one station, modestly increased collision pressure on others (tempered by CWmax=100), a skew in fairness compared to the baseline, and largely unchanged aggregate throughput until the medium saturates.


#### 6.5.1 Load the Modified XML Scenario File


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO04="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_Station10only.xml"
!echo Modified scenario is $SCENARIO04

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO04, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO04, lineterm='')

#for line in diff:
#    print(line)

#### 6.5.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO04

Total Traffic:

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_Station10only"):
    os.chdir("results_MobComp_A04_modified_Station10only")

print("Total:")
result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)
plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay,180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

Station 10 (with modified MAC parameters) in comparison to Station 8 (no modifications):

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_Station10only"):
    os.chdir("results_MobComp_A04_modified_Station10only")

print("Station 10 (with modification) and Station 08 (similar to all others):")
result_sta10_offer = pd.read_csv('offer_SA10_DA1_AC2_ID_140_data_.csv', skiprows=2)
result_sta10_thrp = pd.read_csv ('thrp_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta10_delay = pd.read_csv('delay_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta08_offer = pd.read_csv('offer_SA8_DA1_AC2_ID_112_data_.csv', skiprows=2)
result_sta08_thrp = pd.read_csv ('thrp_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)
result_sta08_delay = pd.read_csv('delay_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)

plot_thrp_offer_delay(result_sta10_offer, result_sta10_thrp, result_sta10_delay,2)
plot_thrp_offer_delay(result_sta08_offer, result_sta08_thrp, result_sta08_delay,2)

os.chdir(ROOT_DIR)

### 6.6 Performance with Modified PHY Parameter in Station 10
The OFDM-based PHY modes in IEEE 802.11 start with 802.11a, which introduced OFDM in the 5 GHz band with 20 MHz channels and data rates up to 54 Mb/s using modulation schemes up to 64-QAM. 802.11g followed, bringing OFDM into the 2.4 GHz band while keeping backward compatibility with 802.11b, offering similar rates and channelization. 802.11n introduced the HT (High Throughput) OFDM mode, adding MIMO spatial streams, optional 40 MHz channel bonding, and higher coding rates, increasing throughput into the hundreds of Mb/s. 802.11ac then defined the VHT (Very High Throughput) OFDM mode in 5 GHz, supporting 80 and 160 MHz channels, more spatial streams, 256-QAM modulation, and multi-user MIMO, which allowed multi-gigabit rates. 802.11ax extended OFDM into HE (High Efficiency) mode, applying OFDMA for dividing subcarriers among multiple users, 1024-QAM, improved spatial reuse, and operation across 2.4, 5, and 6 GHz bands, making it more efficient in dense deployments. The newest generation, 802.11be, introduces the EHT (Extremely High Throughput) OFDM mode, with up to 320 MHz channels in 6 GHz, 4096-QAM modulation, multi-link operation across different bands, and expanded MU-MIMO capabilities, pushing Wi-Fi performance to tens of gigabits per second.

The new XML Scenario file is [ETH_Course/MobComp_assignment04_modified_phy.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_phy.xml).
The simulation results will be stored in 
"[results_MobComp_A04_modified_phy](results_MobComp_A04_modified_phy)".

>Same topology and run length as before: one fixed receiver at (0,0) and 36 mobile uplink senders placed on a ~6 m ring, each starting one second apart from 2 s to 37 s, over a 50.001 s emulation with 0.5 s stats to `results_MobComp_A04_modified_phy`. PHY/MAC parameters stay on 2.4 GHz channel 6 at 10 dBm with omni antennas, standard 802.11a/g rates (BER tables provided), RTS/CTS effectively off (threshold 2500 B), and fragmentation active for 1200-byte UDP packets (threshold 512 B). All stations use EDCA AC=2 with AIFSN=4, CWmin=5, CWmax=100 and `phymode_54Mbps`, mean load 0.75 Mb/s, except the node with MACAddress 10 at ~10 s near (1.04, 5.91), which is intentionally PHY-limited to `phymode_6Mbps` and drives a higher 4.0 Mb/s offered load; three senders (start 8 s, 10 s, 11 s) have delay/offer evaluation enabled, otherwise evaluation is throughput-centric. Functionally this “modified_phy” scenario provokes the classic 802.11 rate-anomaly behavior: the slow-rate, high-offered-load station soaks disproportionate airtime, depressing peers’ throughput while its own throughput is capped by 6 Mb/s PHY and contention, shifting fairness and aggregate efficiency compared to the all-54 Mb/s baseline.


#### 6.6.1 Load the Modified XML Scenario File


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO05="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_phy.xml"
!echo Modified scenario is $SCENARIO05

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO05, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO05, lineterm='')

#for line in diff:
#    print(line)

#### 6.6.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO05

Total Traffic:

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_phy"):
    os.chdir("results_MobComp_A04_modified_phy")

print("Total:")
result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)
plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay,180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

Station 10 (with modified PHY parameter and traffic load) in comparison to Station 8 (no modifications):

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_phy"):
    os.chdir("results_MobComp_A04_modified_phy")

print("Station 10 (with modification) and Station 08 (similar to all others):")
result_sta10_offer = pd.read_csv('offer_SA10_DA1_AC2_ID_140_data_.csv', skiprows=2)
result_sta10_thrp = pd.read_csv ('thrp_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta10_delay = pd.read_csv('delay_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta08_offer = pd.read_csv('offer_SA8_DA1_AC2_ID_112_data_.csv', skiprows=2)
result_sta08_thrp = pd.read_csv ('thrp_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)
result_sta08_delay = pd.read_csv('delay_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)

plot_thrp_offer_delay(result_sta10_offer, result_sta10_thrp, result_sta10_delay,5)
plot_thrp_offer_delay(result_sta08_offer, result_sta08_thrp, result_sta08_delay,5)

os.chdir(ROOT_DIR)

----
### 6.7 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**In the first simulation, section 2.2, why does the resulting throughput decrease so strongly at around 20 Mb/s (see the white line)?** Answer:

(add your answer here)

**In section 5.2, only station 10 is modified, all other stations remain configured as in section 2.2. The graphs show that station 10 has now a considerable advantage. Explain why.** Answer:

(add your answer here)

**Explain the results of section 6.2: How do total throughput and total delay change with the modified PHY modes?** Answer:

(add your answer here)

**Is there an advantage for station 10 to operate with a higher PHY mode, while other stations operate with lower modes?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)